# Preparación de Fantastik Break para evaluar Pix2Vox++ sobre tazas rotas

Este notebook prepara un **benchmark externo de tazas rotas** para la fase E2.

El objetivo **no es reparar todavía la taza**. Aquí queremos comprobar:

> Si Pix2Vox++ recibe cinco imágenes de una taza rota, ¿es capaz de reconstruir correctamente la geometría rota que está observando?

Se utilizan únicamente los archivos `*_roto.npy` de Fantastik Break, cada uno con una nube de puntos `(2048, 3)`.

El flujo será:

1. Auditar las 61 nubes rotas.
2. Comprobar escala y orientación.
3. Aplicar una transformación global fija al sistema de render de Pix2Vox++.
4. Reconstruir una superficie conservadora con Ball Pivoting (BPA).
5. Comprobar visual y cuantitativamente que BPA no rellena artificialmente la rotura.
6. Generar exactamente las cinco vistas `00`, `05`, `10`, `14` y `19`.
7. Validar primero un piloto de cinco objetos.
8. Preparar las 61 muestras solo cuando el piloto sea correcto.
9. Crear un catálogo final para evaluar checkpoints de Pix2Vox++ sin reentrenarlos.
10. Añadir un puente `BINVOX → NPY` para la futura fase E3.

Los `*_completo.npy` no se utilizan en esta evaluación E2; se reservan para E3.

## 1. Montar Drive e instalar dependencias

Se usa Open3D para BPA, Trimesh para mallas y muestreo de superficie, BlenderProc para mantener el estilo de render del dataset E2 y SciPy para las métricas de fidelidad.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip -q install -U pip

%pip -q install open3d==0.19.0
%pip -q install trimesh scipy pandas matplotlib plotly imageio blenderproc scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Configuración

Las cinco vistas son exactamente las empleadas por los experimentos de cinco vistas de Pix2Vox++: `[0, 5, 10, 14, 19]`.

El render conserva los parámetros del pipeline E2: `224×224`, focal de `50 mm`, radio de cámara `2.2`, objetivo `[0.5,0.5,0.5]`, material gris y la misma iluminación.

In [ ]:
from pathlib import Path
import json, os, random, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from PIL import Image
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
import open3d as o3d
import trimesh
import torch
from scipy.ndimage import binary_closing, label
from skimage.measure import marching_cubes

RUTA_BASE = Path('/content/drive/MyDrive/Datos_E2_E3')
RUTA_FANTASTIK = RUTA_BASE / 'General/Fantastik_Break_Preprocesado'
RUTA_SALIDA = RUTA_BASE / 'E2/Fantastik_Break_Pix2Vox'
RUTA_MALLAS_BPA = RUTA_SALIDA / 'mallas_bpa'
RUTA_PUNTOS_RENDER = RUTA_SALIDA / 'puntos_render'
RUTA_RENDERS = RUTA_SALIDA / 'renders_5v'
RUTA_INFORMES = RUTA_SALIDA / 'informes'
for r in [RUTA_SALIDA, RUTA_MALLAS_BPA, RUTA_PUNTOS_RENDER, RUTA_RENDERS, RUTA_INFORMES]:
    r.mkdir(parents=True, exist_ok=True)

SEMILLA = 2026
random.seed(SEMILLA)
np.random.seed(SEMILLA)

VISTAS_5 = [
    {'view_id':0,  'filename':'00.png', 'azimuth_deg':0.0,   'elevation_deg':20.0},
    {'view_id':5,  'filename':'05.png', 'azimuth_deg':225.0, 'elevation_deg':20.0},
    {'view_id':10, 'filename':'10.png', 'azimuth_deg':90.0,  'elevation_deg':40.0},
    {'view_id':14, 'filename':'14.png', 'azimuth_deg':270.0, 'elevation_deg':40.0},
    {'view_id':19, 'filename':'19.png', 'azimuth_deg':270.0, 'elevation_deg':60.0},
]
RADIO_CAMARA = 2.2
DISTANCIA_FOCAL_MM = 50.0
ANCHO_IMAGEN = 224
ALTO_IMAGEN = 224

print('Entrada:', RUTA_FANTASTIK)
print('Salida:', RUTA_SALIDA)
print('Vistas:', [v['view_id'] for v in VISTAS_5])

Entrada: /content/drive/MyDrive/Datos_E2_E3/General/Fantastik_Break_Preprocesado
Salida: /content/drive/MyDrive/Datos_E2_E3/E2/Fantastik_Break_Pix2Vox
Vistas: [0, 5, 10, 14, 19]


## 3. Auditoría estructural de Fantastik Break

Se comprueba que todos los `*_roto.npy` tengan forma `(2048,3)`, valores finitos y, únicamente como control del dataset, que exista su pareja completa.

In [ ]:
archivos_rotos = sorted(RUTA_FANTASTIK.rglob('*_roto.npy'))
archivos_completos = sorted(RUTA_FANTASTIK.rglob('*_completo.npy'))

def quitar_sufijo(ruta, sufijo):
    return ruta.stem[:-len(sufijo)]

mapa_completos = {quitar_sufijo(r, '_completo'): r for r in archivos_completos}
registros=[]
for ruta in archivos_rotos:
    identificador = quitar_sufijo(ruta, '_roto')
    arr = np.load(ruta, mmap_mode='r', allow_pickle=False)
    registros.append({
        'identificador': identificador,
        'ruta_roto': str(ruta),
        'ruta_completo': str(mapa_completos[identificador]) if identificador in mapa_completos else None,
        'shape': tuple(arr.shape),
        'dtype': str(arr.dtype),
        'todo_finito': bool(np.isfinite(arr).all()),
        'tiene_completo': identificador in mapa_completos,
    })

df_fantastik = pd.DataFrame(registros)
df_fantastik['shape_correcta'] = df_fantastik['shape'].apply(lambda s: s == (2048,3))
display(df_fantastik.head(10))
print('Rotas:', len(df_fantastik))
print('Shape correcta:', int(df_fantastik['shape_correcta'].sum()))
print('Finitas:', int(df_fantastik['todo_finito'].sum()))
print('Con pareja completa:', int(df_fantastik['tiene_completo'].sum()))

if not df_fantastik['shape_correcta'].all():
    raise RuntimeError('Hay nubes con shape distinto de (2048,3).')
if not df_fantastik['todo_finito'].all():
    raise RuntimeError('Hay NaN o infinitos.')

KeyError: 'shape'

## 4. Diagnóstico de escala

No se normaliza cada taza de forma independiente. Una taza muy rota podría tener un bounding box menor y sería agrandada artificialmente.

Primero se comprueba si Fantastik ya comparte un rango aproximadamente `[-1,1]`.

In [ ]:
registros=[]
for _, fila in tqdm(df_fantastik.iterrows(), total=len(df_fantastik), desc='Coordenadas'):
    p = np.load(fila['ruta_roto'], allow_pickle=False).astype(np.float64)
    mn, mx = p.min(axis=0), p.max(axis=0)
    ext = mx-mn
    registros.append({
        'identificador': fila['identificador'],
        'min_x':mn[0], 'min_y':mn[1], 'min_z':mn[2],
        'max_x':mx[0], 'max_y':mx[1], 'max_z':mx[2],
        'centro_bbox_x':(mn[0]+mx[0])/2, 'centro_bbox_y':(mn[1]+mx[1])/2, 'centro_bbox_z':(mn[2]+mx[2])/2,
        'centroide_x':p[:,0].mean(), 'centroide_y':p[:,1].mean(), 'centroide_z':p[:,2].mean(),
        'extension_x':ext[0], 'extension_y':ext[1], 'extension_z':ext[2], 'dimension_max':ext.max(),
        'fuera_menos1_mas1': int(np.sum((p < -1.0) | (p > 1.0))),
    })

df_normalizacion = pd.DataFrame(registros)
print('Mínimos globales:', df_normalizacion[['min_x','min_y','min_z']].min().round(6).to_dict())
print('Máximos globales:', df_normalizacion[['max_x','max_y','max_z']].max().round(6).to_dict())
print('Puntos fuera de [-1,1]:', int(df_normalizacion['fuera_menos1_mas1'].sum()))
print('\nCentros:')
display(df_normalizacion[['centro_bbox_x','centro_bbox_y','centro_bbox_z','centroide_x','centroide_y','centroide_z']].describe().round(4))
print('\nExtensiones:')
display(df_normalizacion[['extension_x','extension_y','extension_z','dimension_max']].describe().round(4))

## 5. Visualización de una nube rota original

Se muestra una nube antes de transformarla. Hay que comprobar visualmente qué eje es vertical y si la taza tiene una orientación razonable.

In [ ]:
fila_ejemplo = df_fantastik.sample(n=1, random_state=SEMILLA).iloc[0]
ID_EJEMPLO = fila_ejemplo['identificador']
p_original = np.load(fila_ejemplo['ruta_roto'], allow_pickle=False).astype(np.float64)

fig = go.Figure([go.Scatter3d(x=p_original[:,0], y=p_original[:,1], z=p_original[:,2], mode='markers', marker={'size':2})])
fig.update_layout(title=f'Fantastik roto original — {ID_EJEMPLO}', scene={'xaxis_title':'X','yaxis_title':'Y','zaxis_title':'Z','aspectmode':'data'}, width=850, height=750, showlegend=False)
fig.show()

fig2, ax = plt.subplots(1,3,figsize=(15,5))
ax[0].scatter(p_original[:,0],p_original[:,1],s=2); ax[0].set_title('XY'); ax[0].set_xlabel('X'); ax[0].set_ylabel('Y'); ax[0].axis('equal')
ax[1].scatter(p_original[:,0],p_original[:,2],s=2); ax[1].set_title('XZ'); ax[1].set_xlabel('X'); ax[1].set_ylabel('Z'); ax[1].axis('equal')
ax[2].scatter(p_original[:,1],p_original[:,2],s=2); ax[2].set_title('YZ'); ax[2].set_xlabel('Y'); ax[2].set_ylabel('Z'); ax[2].axis('equal')
plt.tight_layout(); plt.show()

## 6. Transformación global al cubo de render `[0,1]³`

El renderer original de ShapeNet apunta al centro `[0.5,0.5,0.5]`. Fantastik está alrededor del origen, por lo que proponemos una transformación fija:

```text
p_render = (p_orientado + 1) / 2
```

`MATRIZ_ORIENTACION` se deja inicialmente como identidad. Solo debe modificarse si la inspección visual demuestra una diferencia global de ejes.

In [ ]:
MATRIZ_ORIENTACION = np.eye(3, dtype=np.float64)

# Ejemplo si visualmente hubiera que intercambiar Y y Z:
# MATRIZ_ORIENTACION = np.array([[1,0,0],[0,0,1],[0,1,0]], dtype=np.float64)

def validar_matriz(M):
    M=np.asarray(M,dtype=np.float64)
    if M.shape != (3,3) or not np.allclose(M@M.T, np.eye(3), atol=1e-8):
        raise ValueError('MATRIZ_ORIENTACION debe ser ortogonal 3x3.')
validar_matriz(MATRIZ_ORIENTACION)

def fantastik_a_render(puntos):
    p=np.asarray(puntos,dtype=np.float64)
    p_orientado = p @ MATRIZ_ORIENTACION.T
    return (p_orientado + 1.0) / 2.0

p_render = fantastik_a_render(p_original)
print('Min:',p_render.min(axis=0)); print('Max:',p_render.max(axis=0))
print('Puntos fuera [0,1]:', int(np.any((p_render<0)|(p_render>1),axis=1).sum()))

fig = go.Figure([go.Scatter3d(x=p_render[:,0], y=p_render[:,1], z=p_render[:,2], mode='markers', marker={'size':2})])
fig.update_layout(title=f'En coordenadas de render — {ID_EJEMPLO}', scene={'xaxis':{'range':[0,1]},'yaxis':{'range':[0,1]},'zaxis':{'range':[0,1]},'aspectmode':'cube'}, width=850,height=750,showlegend=False)
fig.show()

## 7. Reconstrucción conservadora de superficie con BPA

Pix2Vox++ fue entrenado con superficies, no con puntos aislados. Se reconstruye una malla con Ball Pivoting.

No se usa Poisson, `convex hull`, cierre de agujeros ni reparación watertight, porque podrían completar la zona rota antes de la inferencia.

In [ ]:
# ============================================================
# 7. TEMPLATE COMPLETA V3
#
# COMPARAMOS:
#
# C3_poisson_bruto
#     Poisson casi sin recorte
#
# C4_BPA_completa
#     BPA sobre la nube COMPLETA + cierre moderado
#
# C5_BPA_agresiva
#     BPA sobre la nube COMPLETA con radios mayores
#     + cierre más agresivo
#
# Después aplicamos la nube ROTA como máscara.
#
# CAMBIO IMPORTANTE:
# el umbral de la rotura utiliza la SEPARACIÓN DE LA NUBE
# COMPLETA, no la de la rota.
# ============================================================

from scipy.spatial import cKDTree


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

MODOS_TEMPLATE_V3 = [
    "C3_poisson_bruto",
    "C4_BPA_completa",
    "C5_BPA_agresiva",
]


# ------------------------------------------------------------
# Poisson
# ------------------------------------------------------------

POISSON_DEPTH_V3 = 10
POISSON_SMOOTH_ITER = 8


# ------------------------------------------------------------
# BPA completo
# ------------------------------------------------------------

RADIOS_BPA_C4 = [
    1.5,
    2.5,
    4.0,
    6.0,
]

RADIOS_BPA_C5 = [
    2.0,
    4.0,
    6.0,
    9.0,
]


# ------------------------------------------------------------
# Cierre de agujeros de la COMPLETA
#
# Aquí sí podemos cerrar pequeños agujeros con bastante libertad:
# no existen roturas artificiales todavía.
#
# Hay que evitar cerrar:
# - boca de la taza;
# - hueco grande del asa.
# ------------------------------------------------------------

HOLE_SIZE_C4_FRAC_OBJETO = 0.045
HOLE_SIZE_C5_FRAC_OBJETO = 0.075


# ------------------------------------------------------------
# Suavizado de BPA
# ------------------------------------------------------------

BPA_SMOOTH_C4 = 2
BPA_SMOOTH_C5 = 4


# ============================================================
# UMBRALES DE ROTURA
#
# Antes el máximo era 18.
# Ahora probamos también 25.
# ============================================================

CONFIGURACIONES_ROTURA_V3 = {
    "R12": 12.0,
    "R18": 18.0,
    "R25": 25.0,
}


# ============================================================
# SUAVIZADO DE LA MÁSCARA
# ============================================================

ITERACIONES_SUAVIZADO_MASCARA = 8
PESO_VECINOS_MASCARA = 0.70

MAX_AREA_ISLA_ELIMINADA_FRAC = 0.003


# ============================================================
# 1. SEPARACIÓN MEDIANA
# ============================================================

def separacion_mediana(puntos):

    puntos = np.asarray(
        puntos,
        dtype=np.float64
    )

    arbol = cKDTree(
        puntos
    )

    distancias, _ = arbol.query(
        puntos,
        k=2
    )

    d = distancias[:, 1]

    d = d[
        np.isfinite(d)
        & (d > 0)
    ]

    if len(d) == 0:
        raise RuntimeError(
            "No se pudo estimar la separación mediana."
        )

    return float(
        np.median(d)
    )


# ============================================================
# 2. CONVERSIÓN OPEN3D ↔ TRIMESH
# ============================================================

def o3d_a_trimesh(mesh_o3d):

    vertices = np.asarray(
        mesh_o3d.vertices,
        dtype=np.float64
    )

    caras = np.asarray(
        mesh_o3d.triangles,
        dtype=np.int64
    )

    if (
        len(vertices) == 0
        or len(caras) == 0
    ):
        raise RuntimeError(
            "Malla Open3D vacía."
        )

    return trimesh.Trimesh(
        vertices=vertices,
        faces=caras,
        process=False
    )


def trimesh_a_tensor_o3d(mesh):

    vertices = np.asarray(
        mesh.vertices,
        dtype=np.float32
    )

    caras = np.asarray(
        mesh.faces,
        dtype=np.int64
    )

    return o3d.t.geometry.TriangleMesh(
        vertex_positions=o3d.core.Tensor(
            vertices,
            dtype=o3d.core.Dtype.Float32
        ),
        triangle_indices=o3d.core.Tensor(
            caras,
            dtype=o3d.core.Dtype.Int64
        )
    )


# ============================================================
# 3. CIERRE DE AGUJEROS EN LA COMPLETA
# ============================================================

def cerrar_huecos_completa(
    mesh,
    fraccion_hole_size
):

    vertices = np.asarray(
        mesh.vertices,
        dtype=np.float64
    )

    dimension_max = float(
        np.ptp(
            vertices,
            axis=0
        ).max()
    )

    hole_size = float(
        fraccion_hole_size
        * dimension_max
    )

    mesh_tensor = (
        trimesh_a_tensor_o3d(
            mesh
        )
    )

    try:

        mesh_tensor = (
            mesh_tensor.fill_holes(
                hole_size=hole_size
            )
        )

    except Exception as exc:

        print(
            "[AVISO] fill_holes no aplicado:",
            exc
        )

        return mesh


    legacy = (
        mesh_tensor.to_legacy()
    )

    legacy.remove_duplicated_vertices()
    legacy.remove_duplicated_triangles()
    legacy.remove_degenerate_triangles()
    legacy.remove_unreferenced_vertices()


    return o3d_a_trimesh(
        legacy
    )


# ============================================================
# 4. TEMPLATE C3: POISSON BRUTO
# ============================================================

def reconstruir_completa_poisson_bruto(
    puntos
):

    puntos = np.asarray(
        puntos,
        dtype=np.float64
    )

    sep = separacion_mediana(
        puntos
    )


    pc = o3d.geometry.PointCloud()

    pc.points = (
        o3d.utility.Vector3dVector(
            puntos
        )
    )


    pc.estimate_normals(
        search_param=(
            o3d.geometry
            .KDTreeSearchParamHybrid(
                radius=max(
                    8.0 * sep,
                    1e-4
                ),
                max_nn=80
            )
        )
    )


    try:

        pc.orient_normals_consistent_tangent_plane(
            60
        )

    except Exception:
        pass


    mesh_o3d, _ = (
        o3d.geometry.TriangleMesh
        .create_from_point_cloud_poisson(
            pc,
            depth=POISSON_DEPTH_V3,
            scale=1.03,
            linear_fit=False
        )
    )


    # --------------------------------------------------------
    # Suavizado
    # --------------------------------------------------------

    try:

        mesh_o3d = (
            mesh_o3d
            .filter_smooth_taubin(
                number_of_iterations=
                    POISSON_SMOOTH_ITER,
                lambda_filter=0.5,
                mu=-0.53
            )
        )

    except Exception:
        pass


    mesh_o3d.remove_duplicated_vertices()
    mesh_o3d.remove_duplicated_triangles()
    mesh_o3d.remove_degenerate_triangles()
    mesh_o3d.remove_unreferenced_vertices()


    mesh = o3d_a_trimesh(
        mesh_o3d
    )


    # --------------------------------------------------------
    # ÚNICAMENTE recorte amplio al bbox.
    #
    # NO filtramos por distancia.
    # NO filtramos por densidad.
    # --------------------------------------------------------

    minimo = (
        puntos.min(axis=0)
        -
        6.0 * sep
    )

    maximo = (
        puntos.max(axis=0)
        +
        6.0 * sep
    )


    centroides = np.asarray(
        mesh.triangles_center,
        dtype=np.float64
    )


    dentro = np.all(
        (
            centroides >= minimo
        )
        &
        (
            centroides <= maximo
        ),
        axis=1
    )


    mesh.update_faces(
        dentro
    )

    mesh.remove_unreferenced_vertices()


    return (
        mesh,
        sep
    )


# ============================================================
# 5. TEMPLATE BPA
# ============================================================

def reconstruir_completa_bpa(
    puntos,
    factores_radio,
    hole_size_frac,
    iteraciones_suavizado
):

    puntos = np.asarray(
        puntos,
        dtype=np.float64
    )

    sep = separacion_mediana(
        puntos
    )


    # --------------------------------------------------------
    # Point cloud
    # --------------------------------------------------------

    pc = o3d.geometry.PointCloud()

    pc.points = (
        o3d.utility.Vector3dVector(
            puntos
        )
    )


    pc.estimate_normals(
        search_param=(
            o3d.geometry
            .KDTreeSearchParamHybrid(
                radius=max(
                    6.0 * sep,
                    1e-4
                ),
                max_nn=60
            )
        )
    )


    try:

        pc.orient_normals_consistent_tangent_plane(
            50
        )

    except Exception:
        pass


    # --------------------------------------------------------
    # BPA
    # --------------------------------------------------------

    radios = o3d.utility.DoubleVector(
        [
            sep * factor
            for factor
            in factores_radio
        ]
    )


    mesh_o3d = (
        o3d.geometry.TriangleMesh
        .create_from_point_cloud_ball_pivoting(
            pc,
            radios
        )
    )


    mesh_o3d.remove_duplicated_vertices()
    mesh_o3d.remove_duplicated_triangles()
    mesh_o3d.remove_degenerate_triangles()
    mesh_o3d.remove_unreferenced_vertices()


    mesh = o3d_a_trimesh(
        mesh_o3d
    )


    # --------------------------------------------------------
    # Ahora que la taza todavía está COMPLETA podemos cerrar
    # agujeros pequeños producidos por BPA.
    # --------------------------------------------------------

    mesh = cerrar_huecos_completa(
        mesh,
        fraccion_hole_size=
            hole_size_frac
    )


    # --------------------------------------------------------
    # Suavizado Taubin ligero.
    #
    # No cambia qué partes existen:
    # únicamente suaviza los vértices.
    # --------------------------------------------------------

    if (
        iteraciones_suavizado
        > 0
    ):

        mesh_legacy = (
            o3d.geometry.TriangleMesh()
        )


        mesh_legacy.vertices = (
            o3d.utility.Vector3dVector(
                np.asarray(
                    mesh.vertices,
                    dtype=np.float64
                )
            )
        )


        mesh_legacy.triangles = (
            o3d.utility.Vector3iVector(
                np.asarray(
                    mesh.faces,
                    dtype=np.int32
                )
            )
        )


        try:

            mesh_legacy = (
                mesh_legacy
                .filter_smooth_taubin(
                    number_of_iterations=
                        iteraciones_suavizado,
                    lambda_filter=0.5,
                    mu=-0.53
                )
            )

            mesh = o3d_a_trimesh(
                mesh_legacy
            )

        except Exception as exc:

            print(
                "[AVISO] Suavizado BPA:",
                exc
            )


    return (
        mesh,
        sep
    )


# ============================================================
# 6. FUNCIÓN ÚNICA PARA CREAR TEMPLATE
# ============================================================

def reconstruir_template_completa(
    puntos,
    modo
):

    if modo == "C3_poisson_bruto":

        return reconstruir_completa_poisson_bruto(
            puntos
        )


    elif modo == "C4_BPA_completa":

        return reconstruir_completa_bpa(
            puntos=puntos,
            factores_radio=
                RADIOS_BPA_C4,
            hole_size_frac=
                HOLE_SIZE_C4_FRAC_OBJETO,
            iteraciones_suavizado=
                BPA_SMOOTH_C4
        )


    elif modo == "C5_BPA_agresiva":

        return reconstruir_completa_bpa(
            puntos=puntos,
            factores_radio=
                RADIOS_BPA_C5,
            hole_size_frac=
                HOLE_SIZE_C5_FRAC_OBJETO,
            iteraciones_suavizado=
                BPA_SMOOTH_C5
        )


    else:

        raise ValueError(
            f"Modo no reconocido: {modo}"
        )


# ============================================================
# 7. SUAVIZAR SCORE ENTRE CARAS
# ============================================================

def suavizar_valores_caras(
    mesh,
    valores
):

    valores = np.asarray(
        valores,
        dtype=np.float64
    ).copy()


    adyacencia = np.asarray(
        mesh.face_adjacency,
        dtype=np.int64
    )


    if len(adyacencia) == 0:

        return valores


    for _ in range(
        ITERACIONES_SUAVIZADO_MASCARA
    ):

        suma = valores.copy()

        cuenta = np.ones(
            len(valores),
            dtype=np.float64
        )


        np.add.at(
            suma,
            adyacencia[:, 0],
            valores[
                adyacencia[:, 1]
            ]
        )


        np.add.at(
            cuenta,
            adyacencia[:, 0],
            1
        )


        np.add.at(
            suma,
            adyacencia[:, 1],
            valores[
                adyacencia[:, 0]
            ]
        )


        np.add.at(
            cuenta,
            adyacencia[:, 1],
            1
        )


        promedio = (
            suma / cuenta
        )


        valores = (
            (
                1.0
                -
                PESO_VECINOS_MASCARA
            )
            *
            valores
            +
            PESO_VECINOS_MASCARA
            *
            promedio
        )


    return valores


# ============================================================
# 8. RECUPERAR PEQUEÑAS ISLAS ELIMINADAS
# ============================================================

def rellenar_islas_eliminadas_pequenas(
    mesh,
    mantener
):

    mantener = np.asarray(
        mantener,
        dtype=bool
    ).copy()


    indices_eliminados = np.flatnonzero(
        ~mantener
    )


    if len(
        indices_eliminados
    ) == 0:

        return mantener


    conjunto = set(
        int(x)
        for x
        in indices_eliminados
    )


    vecinos = {
        int(x): []
        for x
        in indices_eliminados
    }


    for a, b in np.asarray(
        mesh.face_adjacency,
        dtype=np.int64
    ):

        a = int(a)
        b = int(b)

        if (
            a in conjunto
            and
            b in conjunto
        ):

            vecinos[a].append(b)
            vecinos[b].append(a)


    area_faces = np.asarray(
        mesh.area_faces,
        dtype=np.float64
    )


    area_maxima = (
        float(
            mesh.area
        )
        *
        MAX_AREA_ISLA_ELIMINADA_FRAC
    )


    visitados = set()


    for inicio in indices_eliminados:

        inicio = int(
            inicio
        )

        if inicio in visitados:
            continue


        stack = [
            inicio
        ]

        componente = []


        while stack:

            actual = stack.pop()

            if actual in visitados:
                continue

            visitados.add(
                actual
            )

            componente.append(
                actual
            )

            stack.extend(
                vecinos[
                    actual
                ]
            )


        area_comp = float(
            area_faces[
                componente
            ].sum()
        )


        if (
            area_comp
            <= area_maxima
        ):

            mantener[
                componente
            ] = True


    return mantener


# ============================================================
# 9. APLICAR LA ROTURA
# ============================================================

def aplicar_rotura_a_template(
    mesh_completa,
    puntos_roto,
    sep_completa,
    factor_umbral
):
    """
    La nube rota decide QUÉ ZONAS de la template sobreviven.

    IMPORTANTE:
    la distancia se normaliza con sep_completa.

    Antes utilizábamos sep_roto, pero una pieza muy rota tiene
    los mismos 2048 puntos concentrados en menos superficie y
    puede tener una separación artificialmente pequeña.
    """

    puntos_roto = np.asarray(
        puntos_roto,
        dtype=np.float64
    )


    centroides = np.asarray(
        mesh_completa.triangles_center,
        dtype=np.float64
    )


    arbol_roto = cKDTree(
        puntos_roto
    )


    distancias = (
        arbol_roto.query(
            centroides,
            k=1
        )[0]
    )


    score = (
        distancias
        /
        max(
            sep_completa,
            1e-12
        )
    )


    # --------------------------------------------------------
    # Suavizar la frontera de la rotura
    # --------------------------------------------------------

    score_suave = (
        suavizar_valores_caras(
            mesh_completa,
            score
        )
    )


    mantener = (
        score_suave
        <= factor_umbral
    )


    # --------------------------------------------------------
    # Evitar pequeñas islas artificiales eliminadas
    # --------------------------------------------------------

    mantener = (
        rellenar_islas_eliminadas_pequenas(
            mesh_completa,
            mantener
        )
    )


    # --------------------------------------------------------
    # Cortar
    # --------------------------------------------------------

    mesh_rota = (
        mesh_completa.copy()
    )


    mesh_rota.update_faces(
        mantener
    )


    mesh_rota.remove_unreferenced_vertices()


    if (
        mesh_rota.is_empty
        or
        len(mesh_rota.faces) == 0
    ):

        raise RuntimeError(
            "La máscara eliminó toda la malla."
        )


    try:

        trimesh.repair.fix_normals(
            mesh_rota,
            multibody=True
        )

    except Exception:
        pass


    return (
        mesh_rota,
        {
            "factor_umbral_rotura":
                float(
                    factor_umbral
                ),

            "sep_completa_usada":
                float(
                    sep_completa
                ),

            "fraccion_caras_conservadas":
                float(
                    len(mesh_rota.faces)
                    /
                    max(
                        len(
                            mesh_completa.faces
                        ),
                        1
                    )
                ),

            "fraccion_area_conservada":
                float(
                    mesh_rota.area
                    /
                    max(
                        mesh_completa.area,
                        1e-12
                    )
                ),

            "score_mediana":
                float(
                    np.median(
                        score_suave
                    )
                ),

            "score_p95":
                float(
                    np.percentile(
                        score_suave,
                        95
                    )
                ),
        }
    )

## 8. Primera malla BPA y superposición visual

La superposición permite detectar fácilmente superficies que BPA haya creado donde no existían puntos.

In [ ]:
# mesh_ejemplo, sep_ejemplo = reconstruir_bpa(p_render)
# print('Separación mediana:',sep_ejemplo)
# print('Vértices:',len(mesh_ejemplo.vertices),'Caras:',len(mesh_ejemplo.faces),'Watertight:',mesh_ejemplo.is_watertight)

# fig=go.Figure()
# fig.add_trace(go.Mesh3d(x=mesh_ejemplo.vertices[:,0],y=mesh_ejemplo.vertices[:,1],z=mesh_ejemplo.vertices[:,2],i=mesh_ejemplo.faces[:,0],j=mesh_ejemplo.faces[:,1],k=mesh_ejemplo.faces[:,2],opacity=0.45,name='BPA'))
# fig.add_trace(go.Scatter3d(x=p_render[:,0],y=p_render[:,1],z=p_render[:,2],mode='markers',marker={'size':2},name='Puntos'))
# fig.update_layout(title=f'Puntos + BPA — {ID_EJEMPLO}',scene={'xaxis':{'range':[0,1]},'yaxis':{'range':[0,1]},'zaxis':{'range':[0,1]},'aspectmode':'cube'},width=850,height=750)
# fig.show()

## 9. Métricas de fidelidad `NPY → BPA`

Estas métricas no evalúan Pix2Vox++; evalúan la transformación intermedia.

Además de Chamfer, se calcula una **fracción de superficie inventada**: puntos muestreados sobre la malla que quedan a más de `3×` la separación mediana de cualquier punto original.

In [ ]:
def metricas_bpa(puntos_originales, mesh, sep, n=10000):
    np.random.seed(SEMILLA)
    puntos_mesh,_=trimesh.sample.sample_surface(mesh,n)
    arbol_o=cKDTree(puntos_originales); arbol_m=cKDTree(puntos_mesh)
    d_mo=arbol_o.query(puntos_mesh,k=1)[0]
    d_om=arbol_m.query(puntos_originales,k=1)[0]
    umbral=3.0*sep
    return {
        'chamfer_bpa': float(d_mo.mean()+d_om.mean()),
        'original_a_malla_media': float(d_om.mean()),
        'malla_a_original_media': float(d_mo.mean()),
        'malla_a_original_p95': float(np.percentile(d_mo,95)),
        'umbral_3x_sep': float(umbral),
        'fraccion_superficie_inventada': float(np.mean(d_mo>umbral)),
        'cobertura_original': float(np.mean(d_om<=umbral)),
    }

display(pd.DataFrame([metricas_bpa(p_render,mesh_ejemplo,sep_ejemplo)]).round(6))

## 10. Piloto reproducible de cinco tazas

Se generan cinco PLY y un CSV de fidelidad. No se renderizan todavía las 61 muestras.

In [ ]:
# ============================================================
# 10. PILOTO V3:
#
# 10 tazas
# ×
# 3 templates completas
# ×
# 3 niveles de rotura
#
# Todo se guarda aparte.
# ============================================================


RUTA_TEMPLATE_V3 = (
    RUTA_SALIDA
    /
    "pruebas_template_v3"
)


RUTA_V3_COMPLETAS = (
    RUTA_TEMPLATE_V3
    /
    "completas"
)


RUTA_V3_ROTAS = (
    RUTA_TEMPLATE_V3
    /
    "rotas"
)


RUTA_V3_RENDERS = (
    RUTA_TEMPLATE_V3
    /
    "renders"
)


for ruta in [
    RUTA_TEMPLATE_V3,
    RUTA_V3_COMPLETAS,
    RUTA_V3_ROTAS,
    RUTA_V3_RENDERS,
]:

    ruta.mkdir(
        parents=True,
        exist_ok=True
    )


for modo in MODOS_TEMPLATE_V3:

    (
        RUTA_V3_COMPLETAS
        /
        modo
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    for rotura in CONFIGURACIONES_ROTURA_V3:

        (
            RUTA_V3_ROTAS
            /
            modo
            /
            rotura
        ).mkdir(
            parents=True,
            exist_ok=True
        )


# ============================================================
# MISMAS 10 TAZAS
# ============================================================

df_piloto_v3 = (
    df_fantastik
    .sample(
        n=min(
            10,
            len(df_fantastik)
        ),
        random_state=SEMILLA
    )
    .sort_values(
        "identificador"
    )
    .reset_index(
        drop=True
    )
)


resultados_completas_v3 = []
resultados_rotas_v3 = []


# ============================================================
# PROCESAMIENTO
# ============================================================

for _, fila in tqdm(
    df_piloto_v3.iterrows(),
    total=len(
        df_piloto_v3
    ),
    desc="Template V3"
):

    identificador = (
        fila[
            "identificador"
        ]
    )


    roto = fantastik_a_render(

        np.load(
            fila[
                "ruta_roto"
            ],
            allow_pickle=False
        ).astype(
            np.float64
        )
    )


    completo = fantastik_a_render(

        np.load(
            fila[
                "ruta_completo"
            ],
            allow_pickle=False
        ).astype(
            np.float64
        )
    )


    for modo in MODOS_TEMPLATE_V3:

        try:

            # =================================================
            # TEMPLATE COMPLETA
            # =================================================

            (
                mesh_completa,
                sep_completa
            ) = reconstruir_template_completa(
                completo,
                modo
            )


            ruta_completa = (

                RUTA_V3_COMPLETAS
                /
                modo
                /
                f"{identificador}.ply"
            )


            mesh_completa.export(
                ruta_completa
            )


            # =================================================
            # MÉTRICAS DE LA COMPLETA
            # =================================================

            mets_completa = metricas_bpa(
                completo,
                mesh_completa,
                sep_completa,
                n=15000
            )


            resultados_completas_v3.append({

                "identificador":
                    identificador,

                "modo_template":
                    modo,

                "ruta_ply_completa":
                    str(
                        ruta_completa
                    ),

                "vertices_completa":
                    int(
                        len(
                            mesh_completa.vertices
                        )
                    ),

                "caras_completa":
                    int(
                        len(
                            mesh_completa.faces
                        )
                    ),

                "separacion_completa":
                    float(
                        sep_completa
                    ),

                "chamfer_completa":
                    mets_completa[
                        "chamfer_bpa"
                    ],

                "superficie_inventada_completa":
                    mets_completa[
                        "fraccion_superficie_inventada"
                    ],

                "cobertura_completa":
                    mets_completa[
                        "cobertura_original"
                    ],
            })


            # =================================================
            # APLICAR R12 / R18 / R25
            # =================================================

            for (
                nombre_rotura,
                factor
            ) in (
                CONFIGURACIONES_ROTURA_V3
                .items()
            ):

                (
                    mesh_rota,
                    stats
                ) = aplicar_rotura_a_template(

                    mesh_completa=
                        mesh_completa,

                    puntos_roto=
                        roto,

                    sep_completa=
                        sep_completa,

                    factor_umbral=
                        factor
                )


                ruta_rota = (

                    RUTA_V3_ROTAS
                    /
                    modo
                    /
                    nombre_rotura
                    /
                    f"{identificador}.ply"
                )


                mesh_rota.export(
                    ruta_rota
                )


                mets_rota = metricas_bpa(
                    roto,
                    mesh_rota,
                    sep_completa,
                    n=15000
                )


                resultados_rotas_v3.append({

                    "identificador":
                        identificador,

                    "modo_template":
                        modo,

                    "rotura":
                        nombre_rotura,

                    "factor_rotura":
                        float(
                            factor
                        ),

                    "ruta_ply":
                        str(
                            ruta_rota
                        ),

                    "ruta_ply_completa":
                        str(
                            ruta_completa
                        ),

                    **stats,

                    "chamfer_rota":
                        mets_rota[
                            "chamfer_bpa"
                        ],

                    "superficie_inventada_rota":
                        mets_rota[
                            "fraccion_superficie_inventada"
                        ],

                    "cobertura_rota":
                        mets_rota[
                            "cobertura_original"
                        ],
                })


        except Exception as exc:

            print(
                "[ERROR]",
                identificador,
                modo,
                type(exc).__name__,
                exc
            )


# ============================================================
# DATAFRAMES
# ============================================================

df_completas_v3 = pd.DataFrame(
    resultados_completas_v3
)


df_rotas_v3 = pd.DataFrame(
    resultados_rotas_v3
)


# ============================================================
# RESUMEN COMPLETAS
# ============================================================

print()
print("=" * 90)
print("CALIDAD DE LAS TEMPLATES COMPLETAS")
print("=" * 90)


display(

    df_completas_v3

    .groupby(
        "modo_template"
    )[
        [
            "chamfer_completa",
            "superficie_inventada_completa",
            "cobertura_completa",
        ]
    ]

    .mean()

    .round(6)
)


# ============================================================
# RESUMEN ROTAS
# ============================================================

print()
print("=" * 90)
print("CALIDAD DE LAS TEMPLATES TRAS APLICAR ROTURA")
print("=" * 90)


display(

    df_rotas_v3

    .groupby(
        [
            "modo_template",
            "rotura",
        ]
    )[
        [
            "fraccion_area_conservada",
            "chamfer_rota",
            "superficie_inventada_rota",
            "cobertura_rota",
        ]
    ]

    .mean()

    .round(6)
)

## 11. Inspección visual de cualquier ejemplo del piloto

Cambia `ID_VISUAL` por cualquiera de los cinco IDs para revisar varios casos.

In [ ]:
# def visualizar_piloto(identificador):
#     fila=df_metricas_piloto.loc[df_metricas_piloto['identificador']==identificador]
#     if fila.empty: raise ValueError('ID no perteneciente al piloto.')
#     fila=fila.iloc[0]
#     p=np.load(fila['ruta_puntos_render'],allow_pickle=False)
#     mesh=trimesh.load(fila['ruta_ply'],force='mesh',process=False)
#     fig=go.Figure()
#     fig.add_trace(go.Mesh3d(x=mesh.vertices[:,0],y=mesh.vertices[:,1],z=mesh.vertices[:,2],i=mesh.faces[:,0],j=mesh.faces[:,1],k=mesh.faces[:,2],opacity=0.45,name='BPA'))
#     fig.add_trace(go.Scatter3d(x=p[:,0],y=p[:,1],z=p[:,2],mode='markers',marker={'size':2},name='Puntos'))
#     fig.update_layout(title=f'Piloto — {identificador}',scene={'xaxis':{'range':[0,1]},'yaxis':{'range':[0,1]},'zaxis':{'range':[0,1]},'aspectmode':'cube'},width=850,height=750)
#     fig.show()

# ID_VISUAL=df_metricas_piloto.iloc[0]['identificador']
# visualizar_piloto(ID_VISUAL)

## 12. Crear el JSON de las cinco cámaras

In [ ]:
RUTA_VISTAS_5_JSON=RUTA_SALIDA/'camera_views_5.json'
with RUTA_VISTAS_5_JSON.open('w',encoding='utf-8') as f: json.dump(VISTAS_5,f,indent=4)
display(pd.DataFrame(VISTAS_5))
print(RUTA_VISTAS_5_JSON)

## 13. Renderer BlenderProc de Fantastik

A diferencia del renderer original, aquí no se aplica `translate/scale` de BINVOX dentro de Blender: la malla BPA ya está en `[0,1]³`.

Se conservan material, cámaras, focal, resolución, transparencia e iluminación del pipeline E2.

In [ ]:
from textwrap import dedent
RUTA_SCRIPT_RENDER=Path('/content/render_5_vistas_fantastik.py')
CODIGO_RENDER=dedent(r"""
import blenderproc as bproc
import argparse, json
from pathlib import Path
import imageio.v3 as iio
import numpy as np

parser=argparse.ArgumentParser()
parser.add_argument('--ply',required=True); parser.add_argument('--views-json',required=True)
parser.add_argument('--output-dir',required=True); parser.add_argument('--metadata-json',required=True)
parser.add_argument('--focal-length',type=float,required=True); parser.add_argument('--width',type=int,required=True)
parser.add_argument('--height',type=int,required=True); parser.add_argument('--radius',type=float,required=True)
args=parser.parse_args()
output=Path(args.output_dir); output.mkdir(parents=True,exist_ok=True)
metadata_path=Path(args.metadata_json); metadata_path.parent.mkdir(parents=True,exist_ok=True)
with open(args.views_json,'r',encoding='utf-8') as f: vistas=json.load(f)

bproc.init()
objetos=bproc.loader.load_obj(args.ply)
if len(objetos)!=1: raise RuntimeError(f'Se esperaba 1 objeto y se cargaron {len(objetos)}')
objeto=objetos[0]
for cara in objeto.get_mesh().polygons: cara.use_smooth=True
material=bproc.material.create('material_neutro')
material.set_principled_shader_value('Base Color',[0.65,0.65,0.65,1.0])
material.set_principled_shader_value('Roughness',0.55)
objeto.replace_materials(material)

bproc.camera.set_intrinsics_from_blender_params(lens=args.focal_length,lens_unit='MILLIMETERS',image_width=args.width,image_height=args.height,clip_start=0.1,clip_end=10.0)
centro=np.array([0.5,0.5,0.5],dtype=np.float64)
metadata_views=[]
for vista in vistas:
    az=np.deg2rad(float(vista['azimuth_deg'])); el=np.deg2rad(float(vista['elevation_deg']))
    pos=centro+np.array([args.radius*np.cos(el)*np.cos(az),args.radius*np.sin(el),args.radius*np.cos(el)*np.sin(az)],dtype=np.float64)
    forward=(centro-pos); forward=forward/np.linalg.norm(forward)
    world_up=np.array([0.,1.,0.],dtype=np.float64)
    right=np.cross(world_up,forward); right=right/np.linalg.norm(right)
    up=np.cross(forward,right); up=up/np.linalg.norm(up)
    rot=np.column_stack((right,up,-forward))
    T=bproc.math.build_transformation_mat(pos,rot)
    bproc.camera.add_camera_pose(T)
    metadata_views.append({'view_id':int(vista['view_id']),'filename':vista['filename'],'azimuth_deg':float(vista['azimuth_deg']),'elevation_deg':float(vista['elevation_deg']),'camera_radius':float(args.radius),'target_xyz':centro.tolist(),'camera_location_xyz':pos.tolist(),'camera_to_world':T.tolist()})

for i,(pos,energia) in enumerate([([2.5,-2.5,3.5],250),([-2.0,1.5,2.5],140),([0.5,2.5,3.0],100)]):
    luz=bproc.types.Light(light_type='POINT',name=f'luz_{i}'); luz.set_location(pos); luz.set_energy(energia); luz.set_radius(0.6)

bproc.renderer.set_output_format(file_format='PNG',color_depth=8,enable_transparency=True)
bproc.renderer.set_max_amount_of_samples(64); bproc.renderer.set_noise_threshold(0.05)
datos=bproc.renderer.render(keys_with_alpha_channel={'colors'})
imagenes=datos['colors']
if len(imagenes)!=len(vistas): raise RuntimeError('Número de renders inesperado.')
for vista,imagen in zip(vistas,imagenes): iio.imwrite(output/vista['filename'],imagen)
K=bproc.camera.get_intrinsics_as_K_matrix()
with open(metadata_path,'w',encoding='utf-8') as f: json.dump({'focal_length_mm':float(args.focal_length),'image_width':int(args.width),'image_height':int(args.height),'target_xyz':centro.tolist(),'world_up_axis':'+Y','intrinsic_matrix_K':K.tolist(),'views':metadata_views},f,indent=4)
print('Render completado:',output)
""")
RUTA_SCRIPT_RENDER.write_text(CODIGO_RENDER,encoding='utf-8')
print(RUTA_SCRIPT_RENDER)

## 14. Renderizar y visualizar las cinco vistas de un ejemplo

Si estas imágenes no se parecen al estilo de ShapeNet usado para entrenar Pix2Vox++, no se debe continuar con las 61.

In [ ]:
# ============================================================
# 14. DIAGNÓSTICO CORRECTO DE TEMPLATE V3
#
# IMPORTANTE:
#
# Hasta ahora generábamos C3_poisson_bruto + R12/R18/R25,
# pero NO las estábamos mostrando en la matriz.
#
# Ahora vamos a comparar directamente:
#
#   C3 Poisson COMPLETA
#   C3 Poisson + R12
#   C3 Poisson + R18
#   C3 Poisson + R25
#
# y dejamos C4/C5 completas únicamente como referencia.
#
# Después mostramos las 5 vistas de:
#
#   C3_poisson_bruto + R25
#
# ============================================================


RUTA_BLENDER_LOCAL = Path(
    "/content/blender"
)


# ============================================================
# 1. VISTA ÚNICA PARA DIAGNÓSTICO
# ============================================================

VISTA_DIAG = [
    v
    for v in VISTAS_5
    if v["view_id"] == 14
]


RUTA_VISTA_DIAG_V3 = (
    RUTA_TEMPLATE_V3
    / "vista_14_C3_diagnostico.json"
)


with RUTA_VISTA_DIAG_V3.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        VISTA_DIAG,
        f,
        indent=4
    )


# ============================================================
# 2. FUNCIÓN GENÉRICA DE RENDER
# ============================================================

def renderizar_v3(
    identificador,
    etiqueta,
    ruta_ply,
    vistas,
    ruta_json,
    sobrescribir=True
):

    carpeta = (
        RUTA_V3_RENDERS
        / etiqueta
        / identificador
    )


    rendering = (
        carpeta
        / "rendering"
    )


    rendering.mkdir(
        parents=True,
        exist_ok=True
    )


    metadata = (
        carpeta
        / "camera_poses.json"
    )


    esperadas = [
        rendering / v["filename"]
        for v in vistas
    ]


    if (
        not sobrescribir
        and all(
            ruta.exists()
            for ruta in esperadas
        )
    ):

        return rendering


    cmd = [

        "blenderproc",
        "run",

        str(
            RUTA_SCRIPT_RENDER
        ),

        "--blender-install-path",
        str(
            RUTA_BLENDER_LOCAL
        ),

        "--ply",
        str(
            ruta_ply
        ),

        "--views-json",
        str(
            ruta_json
        ),

        "--output-dir",
        str(
            rendering
        ),

        "--metadata-json",
        str(
            metadata
        ),

        "--focal-length",
        str(
            DISTANCIA_FOCAL_MM
        ),

        "--width",
        str(
            ANCHO_IMAGEN
        ),

        "--height",
        str(
            ALTO_IMAGEN
        ),

        "--radius",
        str(
            RADIO_CAMARA
        ),
    ]


    env = os.environ.copy()

    env[
        "MPLBACKEND"
    ] = "Agg"


    res = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=env,
        timeout=1200
    )


    if res.returncode != 0:

        raise RuntimeError(
            res.stderr[-3000:]
        )


    n = sum(
        ruta.exists()
        for ruta in esperadas
    )


    if n != len(vistas):

        raise RuntimeError(
            f"{identificador} - {etiqueta}: "
            f"esperadas {len(vistas)} imágenes, obtenidas {n}."
        )


    return rendering


# ============================================================
# 3. COMPROBAR QUE TENEMOS C3 + LAS TRES ROTURAS
# ============================================================

df_c3_roturas = (
    df_rotas_v3[
        df_rotas_v3[
            "modo_template"
        ]
        ==
        "C3_poisson_bruto"
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 90)
print("MÉTRICAS DE C3 POISSON BRUTO + ROTURA")
print("=" * 90)


display(

    df_c3_roturas

    .groupby(
        "rotura"
    )[
        [
            "fraccion_area_conservada",
            "chamfer_rota",
            "superficie_inventada_rota",
            "cobertura_rota",
        ]
    ]

    .mean()

    .round(6)
)


print()
print(
    "Variantes C3 encontradas:",
    sorted(
        df_c3_roturas[
            "rotura"
        ].unique()
    )
)


# ============================================================
# 4. RENDER COMPLETA C3
# ============================================================

rutas_diag_c3 = {}


df_c3_completa = (
    df_completas_v3[
        df_completas_v3[
            "modo_template"
        ]
        ==
        "C3_poisson_bruto"
    ]
    .copy()
)


for _, fila in tqdm(
    df_c3_completa.iterrows(),
    total=len(df_c3_completa),
    desc="C3 completa"
):

    identificador = (
        fila[
            "identificador"
        ]
    )


    etiqueta = (
        "COMPLETA__C3_poisson_bruto"
    )


    carpeta = renderizar_v3(

        identificador=
            identificador,

        etiqueta=
            etiqueta,

        ruta_ply=
            fila[
                "ruta_ply_completa"
            ],

        vistas=
            VISTA_DIAG,

        ruta_json=
            RUTA_VISTA_DIAG_V3,

        sobrescribir=True
    )


    rutas_diag_c3[
        (
            identificador,
            etiqueta
        )
    ] = carpeta


# ============================================================
# 5. RENDER C3 + R12/R18/R25
# ============================================================

for _, fila in tqdm(
    df_c3_roturas.iterrows(),
    total=len(df_c3_roturas),
    desc="C3 roturas"
):

    identificador = (
        fila[
            "identificador"
        ]
    )


    etiqueta = (
        "C3_poisson_bruto"
        + "__"
        + fila[
            "rotura"
        ]
    )


    carpeta = renderizar_v3(

        identificador=
            identificador,

        etiqueta=
            etiqueta,

        ruta_ply=
            fila[
                "ruta_ply"
            ],

        vistas=
            VISTA_DIAG,

        ruta_json=
            RUTA_VISTA_DIAG_V3,

        sobrescribir=True
    )


    rutas_diag_c3[
        (
            identificador,
            etiqueta
        )
    ] = carpeta


# ============================================================
# 6. TAMBIÉN MOSTRAR C4/C5 COMPLETAS COMO REFERENCIA
#
# No las vamos a utilizar como candidato.
# Solo sirven para confirmar visualmente que el problema
# de los agujeros viene de BPA.
# ============================================================

for modo in [
    "C4_BPA_completa",
    "C5_BPA_agresiva",
]:

    df_ref = (
        df_completas_v3[
            df_completas_v3[
                "modo_template"
            ]
            ==
            modo
        ]
    )


    for _, fila in tqdm(
        df_ref.iterrows(),
        total=len(df_ref),
        desc=f"Referencia {modo}"
    ):

        identificador = (
            fila[
                "identificador"
            ]
        )


        etiqueta = (
            "COMPLETA__"
            + modo
        )


        carpeta = renderizar_v3(

            identificador=
                identificador,

            etiqueta=
                etiqueta,

            ruta_ply=
                fila[
                    "ruta_ply_completa"
                ],

            vistas=
                VISTA_DIAG,

            ruta_json=
                RUTA_VISTA_DIAG_V3,

            sobrescribir=False
        )


        rutas_diag_c3[
            (
                identificador,
                etiqueta
            )
        ] = carpeta


# ============================================================
# 7. MATRIZ DIAGNÓSTICA CORRECTA
#
# AHORA SÍ:
#
# COMPLETA C3
# C3 R12
# C3 R18
# C3 R25
# COMPLETA C4
# COMPLETA C5
# ============================================================

ORDEN_C3 = [

    "COMPLETA__C3_poisson_bruto",

    "C3_poisson_bruto__R12",

    "C3_poisson_bruto__R18",

    "C3_poisson_bruto__R25",

    "COMPLETA__C4_BPA_completa",

    "COMPLETA__C5_BPA_agresiva",
]


fig, axes = plt.subplots(

    nrows=len(
        df_piloto_v3
    ),

    ncols=len(
        ORDEN_C3
    ),

    figsize=(
        20,
        3.4
        * len(
            df_piloto_v3
        )
    )
)


if len(
    df_piloto_v3
) == 1:

    axes = np.expand_dims(
        axes,
        axis=0
    )


for fila_idx, (_, fila_id) in enumerate(
    df_piloto_v3.iterrows()
):

    identificador = (
        fila_id[
            "identificador"
        ]
    )


    for col_idx, etiqueta in enumerate(
        ORDEN_C3
    ):

        ax = axes[
            fila_idx,
            col_idx
        ]


        carpeta = (
            rutas_diag_c3.get(
                (
                    identificador,
                    etiqueta
                )
            )
        )


        if carpeta is not None:

            ruta_img = (
                Path(
                    carpeta
                )
                / "14.png"
            )


            if ruta_img.exists():

                ax.imshow(
                    Image.open(
                        ruta_img
                    )
                )


        titulo_corto = (
            etiqueta
            .replace(
                "C3_poisson_bruto",
                "C3"
            )
            .replace(
                "C4_BPA_completa",
                "C4 BPA"
            )
            .replace(
                "C5_BPA_agresiva",
                "C5 BPA agresiva"
            )
            .replace(
                "COMPLETA__",
                "COMPLETA "
            )
            .replace(
                "__",
                " + "
            )
        )


        ax.set_title(
            (
                f"{identificador}\n"
                f"{titulo_corto}"
            ),
            fontsize=8
        )


        ax.axis(
            "off"
        )


plt.suptitle(
    (
        "Fantastik Template V3 — "
        "Poisson completo + máscara de rotura"
    ),
    fontsize=16,
    y=1.001
)


plt.tight_layout()
plt.show()


# ============================================================
# 8. CINCO VISTAS DEL CANDIDATO
#
# Empezamos deliberadamente por el MÁS permisivo.
# ============================================================

CANDIDATO_TEMPLATE = (
    "C3_poisson_bruto"
)

CANDIDATO_ROTURA = (
    "R25"
)


df_candidato_c3 = (

    df_rotas_v3[

        (
            df_rotas_v3[
                "modo_template"
            ]
            ==
            CANDIDATO_TEMPLATE
        )

        &

        (
            df_rotas_v3[
                "rotura"
            ]
            ==
            CANDIDATO_ROTURA
        )
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


rutas_candidato_c3 = {}


for _, fila in tqdm(
    df_candidato_c3.iterrows(),
    total=len(
        df_candidato_c3
    ),
    desc="5 vistas C3 + R25"
):

    identificador = (
        fila[
            "identificador"
        ]
    )


    etiqueta = (
        "FINAL__"
        + CANDIDATO_TEMPLATE
        + "__"
        + CANDIDATO_ROTURA
    )


    carpeta = renderizar_v3(

        identificador=
            identificador,

        etiqueta=
            etiqueta,

        ruta_ply=
            fila[
                "ruta_ply"
            ],

        vistas=
            VISTAS_5,

        ruta_json=
            RUTA_VISTAS_5_JSON,

        sobrescribir=True
    )


    rutas_candidato_c3[
        identificador
    ] = carpeta


# ============================================================
# 9. MATRIZ 10 × 5 DE C3 + R25
# ============================================================

fig, axes = plt.subplots(

    nrows=len(
        df_candidato_c3
    ),

    ncols=5,

    figsize=(
        18,
        3.5
        * len(
            df_candidato_c3
        )
    )
)


if len(
    df_candidato_c3
) == 1:

    axes = np.expand_dims(
        axes,
        axis=0
    )


for fila_idx, (_, fila) in enumerate(
    df_candidato_c3.iterrows()
):

    identificador = (
        fila[
            "identificador"
        ]
    )


    carpeta = Path(
        rutas_candidato_c3[
            identificador
        ]
    )


    for col_idx, vista in enumerate(
        VISTAS_5
    ):

        ax = axes[
            fila_idx,
            col_idx
        ]


        ruta_img = (
            carpeta
            /
            vista[
                "filename"
            ]
        )


        if ruta_img.exists():

            ax.imshow(
                Image.open(
                    ruta_img
                )
            )


        ax.set_title(
            (
                f"{identificador}\n"
                f"{vista['filename']}"
            ),
            fontsize=8
        )


        ax.axis(
            "off"
        )


plt.suptitle(
    (
        "C3 Poisson completo + R25 — "
        "5 vistas candidatas para Pix2Vox++"
    ),
    fontsize=16,
    y=1.001
)


plt.tight_layout()
plt.show()

## 15. Render del piloto de cinco tazas

In [ ]:
# resultados=[]
# for _,fila in tqdm(df_metricas_piloto.iterrows(),total=len(df_metricas_piloto),desc='Render piloto'):
#     resultados.append(renderizar_5v(fila['identificador'],fila['ruta_ply'],sobrescribir=False))
# df_render_piloto=pd.DataFrame(resultados)
# display(df_render_piloto)
# if not df_render_piloto['estado'].isin(['ok','existente_valido']).all(): raise RuntimeError('Algún render del piloto falló.')

In [ ]:
a

## 16. Procesamiento de las 61 tazas rotas

La ejecución está bloqueada por defecto. Solo después de validar visualmente el piloto cambia `PROCESAR_61` a `True`.

El render es reanudable y no vuelve a crear cinco imágenes que ya existan.

In [ ]:
# ============================================================
# 16. REGENERAR DESDE CERO Y PROCESAR LAS 61 TAZAS ROTAS
# ============================================================

import shutil

PROCESAR_61 = True
REGENERAR_DESDE_CERO = True


if not PROCESAR_61:

    print(
        "BLOQUEADO: cambia PROCESAR_61 = True "
        "cuando quieras generar las 61 tazas."
    )

else:

    if REGENERAR_DESDE_CERO:

        print("=" * 75)
        print(
            "LIMPIANDO RESULTADOS ANTERIORES "
            "DE FANTASTIK BREAK"
        )
        print("=" * 75)

        carpetas_a_borrar = [
            RUTA_MALLAS_BPA,
            RUTA_PUNTOS_RENDER,
            RUTA_RENDERS,
            (
                RUTA_SALIDA
                / "predicciones_pix2vox_voxel"
            ),
            (
                RUTA_SALIDA
                / "predicciones_pix2vox_malla"
            ),
            (
                RUTA_SALIDA
                / "gt_roto_voxel_32"
            ),
            (
                RUTA_SALIDA
                / "predicciones_pix2vox_pointcloud"
            ),
        ]

        for ruta in carpetas_a_borrar:

            if ruta.exists():

                shutil.rmtree(
                    ruta
                )

                print(
                    "Eliminada:",
                    ruta,
                )

        if RUTA_INFORMES.exists():

            shutil.rmtree(
                RUTA_INFORMES
            )

            print(
                "Eliminada:",
                RUTA_INFORMES,
            )

        for ruta in [
            RUTA_MALLAS_BPA,
            RUTA_PUNTOS_RENDER,
            RUTA_RENDERS,
            RUTA_INFORMES,
        ]:

            ruta.mkdir(
                parents=True,
                exist_ok=True,
            )

        # La sección 12 creó este JSON antes de la limpieza,
        # pero está fuera de las carpetas eliminadas. Aun así,
        # lo recreamos si faltase por cualquier motivo.
        if not RUTA_VISTAS_5_JSON.exists():

            with RUTA_VISTAS_5_JSON.open(
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    VISTAS_5,
                    f,
                    indent=4,
                )

            print(
                "Recreado:",
                RUTA_VISTAS_5_JSON,
            )

        print()
        print(
            "[OK] Limpieza terminada. "
            "Se regenerará todo con la nueva malla."
        )

    resultados = []

    for _, fila in tqdm(
        df_fantastik.iterrows(),
        total=len(df_fantastik),
        desc=(
            "61 tazas — "
            "BPA limpio + renders nuevos"
        ),
    ):

        identificador = (
            fila[
                "identificador"
            ]
        )

        try:

            m = preparar_malla(
                identificador=identificador,
                ruta_roto=(
                    fila[
                        "ruta_roto"
                    ]
                ),
                sobrescribir=True,
            )

            r = renderizar_5v(
                identificador=identificador,
                ruta_ply=(
                    m[
                        "ruta_ply"
                    ]
                ),
                sobrescribir=True,
            )

            resultados.append({
                **m,

                "estado_render":
                    r[
                        "estado"
                    ],

                "n_imagenes":
                    r[
                        "n_imagenes"
                    ],

                "ruta_rendering":
                    r[
                        "ruta_rendering"
                    ],

                "error":
                    r[
                        "error"
                    ],
            })

        except Exception as exc:

            resultados.append({
                "identificador":
                    identificador,

                "estado_render":
                    "error",

                "error":
                    (
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
            })

    df_preparacion_61 = pd.DataFrame(
        resultados
    )

    ruta_reporte = (
        RUTA_INFORMES
        / "fantastik_61_preparacion.csv"
    )

    df_preparacion_61.to_csv(
        ruta_reporte,
        index=False,
    )

    print()
    print("=" * 75)
    print("RESUMEN DE LA REGENERACIÓN")
    print("=" * 75)

    n_ok = int(
        df_preparacion_61[
            "estado_render"
        ]
        .isin(
            [
                "ok",
                "existente_valido",
            ]
        )
        .sum()
    )

    print(
        "Objetos con 5 renders válidos:",
        n_ok,
        "/",
        len(df_preparacion_61),
    )

    if (
        "huecos_micro_rellenados"
        in df_preparacion_61.columns
    ):

        print(
            "Microagujeros rellenados en total:",
            int(
                df_preparacion_61[
                    "huecos_micro_rellenados"
                ]
                .fillna(0)
                .sum()
            ),
        )

        print(
            "Reducción total de bordes frontera:",
            int(
                df_preparacion_61[
                    "reduccion_bordes_frontera"
                ]
                .fillna(0)
                .sum()
            ),
        )

    columnas_mostrar = [
        columna
        for columna in [
            "identificador",
            "huecos_micro_rellenados",
            "bordes_frontera_antes",
            "bordes_frontera_despues",
            "chamfer_bpa",
            "fraccion_superficie_inventada",
            "cobertura_original",
            "estado_render",
            "n_imagenes",
            "error",
        ]
        if columna in df_preparacion_61.columns
    ]

    display(
        df_preparacion_61[
            columnas_mostrar
        ]
    )

    print(
        "Reporte:",
        ruta_reporte,
    )

    if n_ok != len(
        df_preparacion_61
    ):
        raise RuntimeError(
            "No todas las tazas se regeneraron correctamente. "
            "Revisa las filas con estado_render='error'."
        )

## 17. Auditoría final y catálogo para inferencia Pix2Vox++

Cada una de las 61 muestras debe tener un PLY BPA, su nube transformada y las cinco vistas exactas.

In [ ]:
def auditar():
    regs=[]
    for _,fila in df_fantastik.iterrows():
        i=fila['identificador']; ply=RUTA_MALLAS_BPA/f'{i}.ply'; pts=RUTA_PUNTOS_RENDER/f'{i}.npy'; carpeta=RUTA_RENDERS/i/'rendering'
        rutas={v['view_id']:carpeta/v['filename'] for v in VISTAS_5}
        regs.append({'identificador':i,'npy_roto':fila['ruta_roto'],'ply_bpa':str(ply),'puntos_render_npy':str(pts),'vista_00':str(rutas[0]),'vista_05':str(rutas[5]),'vista_10':str(rutas[10]),'vista_14':str(rutas[14]),'vista_19':str(rutas[19]),'ply_ok':ply.exists(),'puntos_render_ok':pts.exists(),'imagenes_5_ok':all(r.exists() for r in rutas.values()),'valido':ply.exists() and pts.exists() and all(r.exists() for r in rutas.values())})
    return pd.DataFrame(regs)

df_auditoria=auditar(); display(df_auditoria.head(10))
print('Esperados:',len(df_auditoria),'Válidos:',int(df_auditoria['valido'].sum()))
ruta_catalogo=RUTA_INFORMES/'catalogo_fantastik_roto_5v.csv'; df_auditoria.to_csv(ruta_catalogo,index=False)
print('Catálogo:',ruta_catalogo)

## 18. Carga del modelo Pix2vox++ para inferencia sobre tazas rotas
Requiere las clases y funciones ya definidas en Pix2vox++_Ampliado.ipynb: EncoderPix2VoxPlusPlusA, DecoderPix2VoxPlusPlusA, MergerPix2VoxPlusPlusA,
RefinerPix2VoxPlusPlusA, Pix2VoxPlusPlusA, cargar_checkpoint_finetuning

In [ ]:
# ============================================================
# 18. PREPARACIÓN DE LA INFERENCIA PIX2VOX++
# ============================================================

import sys
from skimage.measure import marching_cubes
from scipy.ndimage import binary_dilation


# ------------------------------------------------------------
# Cargar catálogo ya generado
# ------------------------------------------------------------

RUTA_CATALOGO = (
    RUTA_INFORMES
    / "catalogo_fantastik_roto_5v.csv"
)

if not RUTA_CATALOGO.exists():
    raise FileNotFoundError(
        f"No existe el catálogo:\n{RUTA_CATALOGO}\n"
        "Hay que haber ejecutado previamente la preparación de las 61 tazas."
    )

df_inferencia = pd.read_csv(RUTA_CATALOGO)

if "valido" in df_inferencia.columns:
    df_inferencia = (
        df_inferencia[
            df_inferencia["valido"] == True
        ]
        .copy()
        .reset_index(drop=True)
    )

print("Tazas disponibles para inferencia:", len(df_inferencia))

if len(df_inferencia) != 61:
    print(
        "AVISO: se esperaban 61 tazas y hay",
        len(df_inferencia)
    )


# ------------------------------------------------------------
# Rutas de salida
# ------------------------------------------------------------

RUTA_PRED_VOXEL = (
    RUTA_SALIDA
    / "predicciones_pix2vox_voxel"
)

RUTA_PRED_MALLA = (
    RUTA_SALIDA
    / "predicciones_pix2vox_malla"
)

RUTA_GT_VOXEL_ROTO = (
    RUTA_SALIDA
    / "gt_roto_voxel_32"
)

RUTA_PRED_POINTCLOUD = (
    RUTA_SALIDA
    / "predicciones_pix2vox_pointcloud"
)

for ruta in [
    RUTA_PRED_VOXEL,
    RUTA_PRED_MALLA,
    RUTA_GT_VOXEL_ROTO,
    RUTA_PRED_POINTCLOUD,
]:
    ruta.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Cargar modelo
# ------------------------------------------------------------

RUTA_NOTEBOOKS = Path(
    "/content/drive/MyDrive/Colab Notebooks"
)

if str(RUTA_NOTEBOOKS) not in sys.path:
    sys.path.append(
        str(RUTA_NOTEBOOKS)
    )

from modelo_pix2vox import (
    Pix2VoxPlusPlusA,
    cargar_checkpoint_finetuning,
    DEVICE,
)


NOMBRE_EXPERIMENTO_INFERENCIA = (
    "exp13_7categorias_5v_finetuning_mejor.pth"
)

RUTA_CHECKPOINT_INFERENCIA = (
    RUTA_BASE
    / "E2/Pix2Vox++/checkpoints"
    / NOMBRE_EXPERIMENTO_INFERENCIA
)

if not RUTA_CHECKPOINT_INFERENCIA.exists():
    raise FileNotFoundError(
        f"No se encuentra:\n"
        f"{RUTA_CHECKPOINT_INFERENCIA}"
    )


modelo_pix2vox = Pix2VoxPlusPlusA(
    usar_refiner=True,
    usar_merger=True,
    usar_pesos_imagenet=False
)


metadata_checkpoint_inferencia = (
    cargar_checkpoint_finetuning(
        modelo=modelo_pix2vox,
        ruta_checkpoint=RUTA_CHECKPOINT_INFERENCIA,
        dispositivo_carga="cpu",
        estricto=True,
    )
)


if (
    metadata_checkpoint_inferencia["claves_ausentes"]
    or metadata_checkpoint_inferencia["claves_inesperadas"]
):
    raise RuntimeError(
        "El checkpoint no coincide con la arquitectura."
    )


modelo_pix2vox = (
    modelo_pix2vox
    .to(DEVICE)
)

modelo_pix2vox.eval()


UMBRAL_VOXEL = float(
    metadata_checkpoint_inferencia[
        "mejor_umbral"
    ]
)

TAMANO_IMAGEN = 224
RESOLUCION_VOXEL = 32

MEDIA_IMAGEN = np.array(
    [0.5, 0.5, 0.5],
    dtype=np.float32
)

DESVIACION_IMAGEN = np.array(
    [0.5, 0.5, 0.5],
    dtype=np.float32
)


COLUMNAS_VISTAS = [
    "vista_00",
    "vista_05",
    "vista_10",
    "vista_14",
    "vista_19",
]


print()
print("Checkpoint:", NOMBRE_EXPERIMENTO_INFERENCIA)
print(
    "IoU validación del checkpoint:",
    metadata_checkpoint_inferencia[
        "mejor_iou_validacion"
    ]
)
print("Threshold:", UMBRAL_VOXEL)
print("Dispositivo:", DEVICE)

## 19. Inferencia: de 5 fotos a nube de puntos reconstruida


In [ ]:
# ============================================================
# 19. FASE 1
#     PIX2VOX++: 5 IMÁGENES → VOXEL 32³
#     + MÉTRICAS 3D DE SUPERFICIE
#     + MALLA PLY PARA VISUALIZACIÓN
# ============================================================

from scipy.ndimage import binary_dilation, binary_erosion


# ------------------------------------------------------------
# Preparación EXACTA de imagen para la inferencia
# ------------------------------------------------------------

def preparar_imagen_pix2vox(
    ruta_imagen,
    color_fondo=(255, 255, 255),
):
    """
    Convierte un PNG RGBA de BlenderProc a la imagen RGB que
    recibe Pix2Vox++.

    IMPORTANTE:
    - se conserva exactamente el encuadre 224x224 del renderer;
    - NO se recorta el objeto;
    - NO se recentra;
    - NO se reescala según la parte rota;
    - únicamente se compone el canal alpha sobre fondo claro.

    De este modo mantenemos la posición, escala y cámaras del
    sistema global [0,1]^3, pero evitamos convertir la zona
    transparente en negro.
    """

    ruta_imagen = Path(ruta_imagen)

    if not ruta_imagen.exists():
        raise FileNotFoundError(ruta_imagen)

    with Image.open(ruta_imagen) as imagen:
        rgba = imagen.convert("RGBA")

    fondo = Image.new(
        "RGBA",
        rgba.size,
        (
            int(color_fondo[0]),
            int(color_fondo[1]),
            int(color_fondo[2]),
            255,
        ),
    )

    compuesta = Image.alpha_composite(
        fondo,
        rgba,
    ).convert("RGB")

    if compuesta.size != (
        TAMANO_IMAGEN,
        TAMANO_IMAGEN,
    ):
        compuesta = compuesta.resize(
            (
                TAMANO_IMAGEN,
                TAMANO_IMAGEN,
            ),
            resample=Image.Resampling.BILINEAR,
        )

    return compuesta


def cargar_imagenes_fila(fila):
    """
    Carga exactamente las cinco vistas del catálogo y las
    transforma al tensor esperado por Pix2Vox++.
    """

    imgs = []

    for columna in COLUMNAS_VISTAS:

        img = preparar_imagen_pix2vox(
            fila[columna]
        )

        arr = (
            np.asarray(
                img,
                dtype=np.float32,
            )
            / 255.0
        )

        arr = (
            arr
            - MEDIA_IMAGEN
        ) / DESVIACION_IMAGEN

        imgs.append(arr)

    tensor = (
        torch
        .from_numpy(
            np.stack(imgs)
        )
        .permute(
            0, 3, 1, 2
        )
        .unsqueeze(0)
        .float()
    )

    return tensor.to(DEVICE)


@torch.inference_mode()
def inferir_voxel_pix2vox(fila):
    """
    Devuelve la salida nativa de Pix2Vox++:
    volumen de probabilidades 32x32x32.
    """

    imgs = cargar_imagenes_fila(
        fila
    )

    salida = modelo_pix2vox(
        imgs
    )

    voxel_prob = (
        salida["volumen_final"]
        .squeeze()
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    forma_esperada = (
        RESOLUCION_VOXEL,
        RESOLUCION_VOXEL,
        RESOLUCION_VOXEL,
    )

    if voxel_prob.shape != forma_esperada:
        raise RuntimeError(
            f"Shape inesperado: {voxel_prob.shape}"
        )

    return voxel_prob


# ------------------------------------------------------------
# Voxel predicho → malla
# ------------------------------------------------------------

def voxel_binario_a_malla(
    volumen_binario
):
    """
    Convierte el voxel 32³ a una malla únicamente para
    visualizar y guardar la reconstrucción 3D.
    """

    volumen = np.asarray(
        volumen_binario,
        dtype=bool,
    )

    if volumen.sum() == 0:
        return None

    volumen_pad = np.pad(
        volumen.astype(np.float32),
        1,
        mode="constant",
    )

    verts, caras, _, _ = marching_cubes(
        volumen_pad,
        level=0.5,
    )

    verts = (
        verts
        - 1.0
        + 0.5
    ) / RESOLUCION_VOXEL

    verts = np.clip(
        verts,
        0.0,
        1.0,
    )

    mesh = trimesh.Trimesh(
        vertices=verts,
        faces=caras,
        process=False,
    )

    return mesh


# ------------------------------------------------------------
# BPA roto → superficie voxel GLOBAL 32³
# ------------------------------------------------------------

def bpa_roto_a_voxel_32(
    ruta_ply
):
    """
    Rasteriza la SUPERFICIE de la malla BPA rota sobre la
    rejilla global [0,1]^3 de 32³.

    No rellena el interior y no cierra la rotura grande.
    La voxelización es determinista: no depende de muestreo
    aleatorio de puntos.
    """

    mesh = trimesh.load(
        ruta_ply,
        force="mesh",
        process=False,
    )

    if not isinstance(
        mesh,
        trimesh.Trimesh,
    ):
        raise TypeError(
            f"No se obtuvo una malla válida: {ruta_ply}"
        )

    if mesh.is_empty:
        raise ValueError(
            f"Malla vacía: {ruta_ply}"
        )

    pitch = (
        1.0
        / float(RESOLUCION_VOXEL)
    )

    voxel_grid = mesh.voxelized(
        pitch=pitch,
        method="subdivide",
    )

    puntos = np.asarray(
        voxel_grid.points,
        dtype=np.float64,
    )

    vertices = np.asarray(
        mesh.vertices,
        dtype=np.float64,
    )

    if len(vertices) > 0:
        puntos = np.vstack(
            [
                puntos,
                vertices,
            ]
        )

    volumen = np.zeros(
        (
            RESOLUCION_VOXEL,
            RESOLUCION_VOXEL,
            RESOLUCION_VOXEL,
        ),
        dtype=bool,
    )

    if len(puntos) == 0:
        return volumen

    puntos = np.clip(
        puntos,
        0.0,
        1.0 - 1e-10,
    )

    indices = np.floor(
        puntos
        * RESOLUCION_VOXEL
    ).astype(np.int32)

    validos = np.all(
        (indices >= 0)
        & (
            indices
            < RESOLUCION_VOXEL
        ),
        axis=1,
    )

    indices = indices[
        validos
    ]

    volumen[
        indices[:, 0],
        indices[:, 1],
        indices[:, 2],
    ] = True

    return volumen


# ------------------------------------------------------------
# Volumen ocupado → superficie voxel
# ------------------------------------------------------------

def extraer_superficie_voxel(
    volumen
):
    """
    Extrae únicamente los voxeles de frontera de un volumen.

    Esto permite comparar la salida volumétrica de Pix2Vox++
    con la representación superficial de la taza rota, evitando
    penalizar al modelo simplemente porque su salida tenga
    interior ocupado y el GT roto procedente de BPA sea superficie.
    """

    volumen = np.asarray(
        volumen,
        dtype=bool,
    )

    if volumen.sum() == 0:
        return volumen.copy()

    erosionado = binary_erosion(
        volumen,
        structure=np.ones(
            (3, 3, 3),
            dtype=bool,
        ),
        border_value=0,
    )

    superficie = (
        volumen
        & ~erosionado
    )

    return superficie


# ------------------------------------------------------------
# Métricas superficie vs superficie
# ------------------------------------------------------------

def metricas_voxel(
    pred_volumen,
    gt_superficie,
):
    """
    Métricas voxel de SUPERFICIE.

    La salida de Pix2Vox++ es volumétrica, mientras que el GT de
    Fantastik que estamos construyendo desde BPA representa una
    superficie abierta. Por eso la comparación se hace entre:

        superficie(predicción) vs superficie(GT roto)

    También se calcula una versión tolerante a 1 voxel, útil a
    resolución 32³.
    """

    pred_volumen = np.asarray(
        pred_volumen,
        dtype=bool,
    )

    gt = np.asarray(
        gt_superficie,
        dtype=bool,
    )

    pred = extraer_superficie_voxel(
        pred_volumen
    )

    inter = int(
        np.logical_and(
            pred,
            gt,
        ).sum()
    )

    union = int(
        np.logical_or(
            pred,
            gt,
        ).sum()
    )

    n_pred = int(
        pred.sum()
    )

    n_gt = int(
        gt.sum()
    )

    iou = (
        inter / union
        if union > 0
        else np.nan
    )

    precision = (
        inter / n_pred
        if n_pred > 0
        else np.nan
    )

    recall = (
        inter / n_gt
        if n_gt > 0
        else np.nan
    )

    dice = (
        2.0 * inter
        / (n_pred + n_gt)
        if (
            n_pred + n_gt
        ) > 0
        else np.nan
    )

    estructura_tol = np.ones(
        (3, 3, 3),
        dtype=bool,
    )

    gt_dilatado = binary_dilation(
        gt,
        structure=estructura_tol,
        iterations=1,
    )

    pred_dilatado = binary_dilation(
        pred,
        structure=estructura_tol,
        iterations=1,
    )

    precision_tol = (
        np.logical_and(
            pred,
            gt_dilatado,
        ).sum()
        / n_pred
        if n_pred > 0
        else np.nan
    )

    recall_tol = (
        np.logical_and(
            gt,
            pred_dilatado,
        ).sum()
        / n_gt
        if n_gt > 0
        else np.nan
    )

    if (
        np.isfinite(precision_tol)
        and np.isfinite(recall_tol)
        and (
            precision_tol
            + recall_tol
        ) > 0
    ):
        f1_tol = (
            2.0
            * precision_tol
            * recall_tol
            / (
                precision_tol
                + recall_tol
            )
        )
    else:
        f1_tol = np.nan

    return {
        "iou_voxel_roto":
            float(iou),

        "dice_voxel_roto":
            float(dice),

        "precision_voxel_roto":
            float(precision),

        "recall_voxel_roto":
            float(recall),

        "precision_voxel_tolerancia_1":
            float(precision_tol),

        "recall_voxel_tolerancia_1":
            float(recall_tol),

        "f1_voxel_tolerancia_1":
            float(f1_tol),

        "voxeles_predichos_volumen":
            int(pred_volumen.sum()),

        "voxeles_predichos_superficie":
            int(n_pred),

        "voxeles_gt_roto":
            int(n_gt),

        "voxeles_interseccion":
            int(inter),
    }


# ------------------------------------------------------------
# Ejecutar las 61
# ------------------------------------------------------------

np.random.seed(
    SEMILLA
)

resultados_voxel = []


for _, fila in tqdm(
    df_inferencia.iterrows(),
    total=len(df_inferencia),
    desc=(
        "Pix2Vox++ — "
        "evaluación de superficie voxel"
    ),
):

    identificador = (
        fila[
            "identificador"
        ]
    )

    try:

        # ====================================================
        # 1. Inferencia real del modelo
        # ====================================================

        voxel_prob = (
            inferir_voxel_pix2vox(
                fila
            )
        )

        voxel_bin = (
            voxel_prob
            >= UMBRAL_VOXEL
        )


        # ====================================================
        # 2. Guardar voxel
        # ====================================================

        ruta_prob = (
            RUTA_PRED_VOXEL
            / f"{identificador}_prob.npy"
        )

        ruta_bin = (
            RUTA_PRED_VOXEL
            / f"{identificador}_bin.npy"
        )

        np.save(
            ruta_prob,
            voxel_prob,
        )

        np.save(
            ruta_bin,
            voxel_bin.astype(np.uint8),
        )


        # ====================================================
        # 3. Crear PLY de la predicción
        # ====================================================

        mesh_pred = (
            voxel_binario_a_malla(
                voxel_bin
            )
        )

        if mesh_pred is None:

            resultados_voxel.append({
                "identificador":
                    identificador,

                "estado":
                    "prediccion_vacia",

                "error":
                    None,
            })

            continue


        ruta_malla_pred = (
            RUTA_PRED_MALLA
            / f"{identificador}.ply"
        )

        mesh_pred.export(
            ruta_malla_pred
        )


        # ====================================================
        # 4. GT roto como superficie voxel 32³
        # ====================================================

        gt_roto = (
            bpa_roto_a_voxel_32(
                fila[
                    "ply_bpa"
                ]
            )
        )

        ruta_gt = (
            RUTA_GT_VOXEL_ROTO
            / f"{identificador}.npy"
        )

        np.save(
            ruta_gt,
            gt_roto.astype(np.uint8),
        )


        # ====================================================
        # 5. Métricas superficie vs superficie
        # ====================================================

        mets = metricas_voxel(
            pred_volumen=voxel_bin,
            gt_superficie=gt_roto,
        )


        resultados_voxel.append({
            "identificador":
                identificador,

            "estado":
                "ok",

            "tipo_evaluacion_voxel":
                "superficie_vs_superficie_32",

            "ruta_voxel_prob":
                str(ruta_prob),

            "ruta_voxel_bin":
                str(ruta_bin),

            "ruta_gt_roto_voxel":
                str(ruta_gt),

            "ruta_malla_pred":
                str(ruta_malla_pred),

            "error":
                None,

            **mets,
        })


    except Exception as exc:

        resultados_voxel.append({
            "identificador":
                identificador,

            "estado":
                "error",

            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        })


df_metricas_voxel = pd.DataFrame(
    resultados_voxel
)


RUTA_METRICAS_VOXEL = (
    RUTA_INFORMES
    / "evaluacion_pix2vox_voxel_tazas_rotas.csv"
)


df_metricas_voxel.to_csv(
    RUTA_METRICAS_VOXEL,
    index=False,
)


print()
print("=" * 75)
print(
    "MÉTRICAS 3D — "
    "SUPERFICIE VOXEL 32³"
)
print("=" * 75)


df_metricas_voxel_ok = (
    df_metricas_voxel[
        df_metricas_voxel[
            "estado"
        ] == "ok"
    ]
    .copy()
)


if len(df_metricas_voxel_ok) > 0:

    display(
        df_metricas_voxel_ok[
            [
                "iou_voxel_roto",
                "dice_voxel_roto",
                "precision_voxel_roto",
                "recall_voxel_roto",
                "precision_voxel_tolerancia_1",
                "recall_voxel_tolerancia_1",
                "f1_voxel_tolerancia_1",
                "voxeles_predichos_volumen",
                "voxeles_predichos_superficie",
                "voxeles_gt_roto",
            ]
        ]
        .describe()
    )


print()
print(
    "Predicciones válidas:",
    len(df_metricas_voxel_ok),
    "/",
    len(df_metricas_voxel),
)

print(
    "CSV:",
    RUTA_METRICAS_VOXEL,
)

print()
print(
    "NOTA: este IoU es superficie-vs-superficie sobre el "
    "benchmark roto y no debe compararse numéricamente con el "
    "IoU volumétrico de ShapeNet usado durante entrenamiento/test."
)

Prueba con un solo objeto:

## 20. Comparación entre la reconstrucción de Pix2Vox++ y la rotura original

Se evalúan las 61 tazas: para cada una se compara la nube de puntos predicha por el modelo (a partir de las 5 vistas) contra la nube `*_roto.npy` original, mediante Chamfer Distance.

Además de la distancia media, se calcula la **fracción de hueco rellenado**: la proporción de puntos predichos que caen lejos de cualquier punto de la rotura real. Un valor alto indica que el modelo está reconstruyendo material donde la taza rota no tenía nada, es decir, "curando" la rotura en vez de reproducirla. Un valor bajo indica que el modelo respeta el hueco.

El resultado se guarda en un CSV para poder analizarlo o compartirlo sin tener que repetir la inferencia.

In [ ]:
# ============================================================
# 20. COMPARACIÓN VISUAL DE 20 CASOS ALEATORIOS
#     5 IMÁGENES EXACTAS QUE RECIBE PIX2VOX++
#     + RECONSTRUCCIÓN 3D
# ============================================================


df_validas_voxel = (
    df_metricas_voxel[
        df_metricas_voxel[
            "estado"
        ] == "ok"
    ]
    .copy()
)


df_20_visual = (
    df_validas_voxel
    .sample(
        n=min(
            20,
            len(df_validas_voxel),
        ),
        random_state=SEMILLA,
    )
    .reset_index(drop=True)
)


print(
    "IDs seleccionados:"
)


display(
    df_20_visual[
        [
            "identificador",
            "iou_voxel_roto",
            "dice_voxel_roto",
            "precision_voxel_roto",
            "recall_voxel_roto",
            "f1_voxel_tolerancia_1",
        ]
    ]
)


for _, resultado in df_20_visual.iterrows():

    identificador = (
        resultado[
            "identificador"
        ]
    )

    fila = (
        df_inferencia[
            df_inferencia[
                "identificador"
            ] == identificador
        ]
        .iloc[0]
    )

    fig = plt.figure(
        figsize=(18, 10)
    )

    gs = fig.add_gridspec(
        2,
        5,
        height_ratios=[
            1,
            2.2,
        ],
        hspace=0.20,
    )

    # --------------------------------------------------------
    # Fila superior: exactamente el RGB que entra al modelo
    # --------------------------------------------------------

    for j, columna in enumerate(
        COLUMNAS_VISTAS
    ):

        ax = fig.add_subplot(
            gs[
                0,
                j,
            ]
        )

        img = preparar_imagen_pix2vox(
            fila[
                columna
            ]
        )

        ax.imshow(
            img
        )

        ax.set_title(
            columna.replace(
                "vista_",
                "Vista ",
            )
        )

        ax.axis(
            "off"
        )

    # --------------------------------------------------------
    # Fila inferior: salida 3D de Pix2Vox++
    # --------------------------------------------------------

    ax3d = fig.add_subplot(
        gs[
            1,
            :,
        ],
        projection="3d",
    )

    mesh = trimesh.load(
        resultado[
            "ruta_malla_pred"
        ],
        force="mesh",
        process=False,
    )

    vertices = np.asarray(
        mesh.vertices
    )

    caras = np.asarray(
        mesh.faces
    )

    ax3d.plot_trisurf(
        vertices[:, 0],
        vertices[:, 2],
        vertices[:, 1],
        triangles=caras,
        linewidth=0.05,
    )

    ax3d.set_xlim(
        0,
        1,
    )

    ax3d.set_ylim(
        0,
        1,
    )

    ax3d.set_zlim(
        0,
        1,
    )

    ax3d.set_box_aspect(
        (
            1,
            1,
            1,
        )
    )

    ax3d.set_xlabel(
        "X"
    )

    ax3d.set_ylabel(
        "Z"
    )

    ax3d.set_zlabel(
        "Y"
    )

    ax3d.view_init(
        elev=20,
        azim=-60,
    )

    ax3d.set_title(
        "Salida 3D de Pix2Vox++"
    )

    fig.suptitle(
        (
            f"{identificador}"
            f"   |   IoU superficie = "
            f"{resultado['iou_voxel_roto']:.4f}"
            f"   |   Dice = "
            f"{resultado['dice_voxel_roto']:.4f}"
            f"   |   F1 tol. 1 voxel = "
            f"{resultado['f1_voxel_tolerancia_1']:.4f}"
        ),
        fontsize=15,
    )

    plt.tight_layout()
    plt.show()

## 21.  Voxel → point cloud + métricas contra roto

In [ ]:
# ============================================================
# 21. FASE 2
#     SALIDA PIX2VOX++ → NUBE DE 2048 PUNTOS
#     COMPARACIÓN CONTRA LA NUBE ROTA REAL
# ============================================================


def separacion_mediana_local(
    puntos
):
    """
    Distancia mediana al vecino más cercano.
    """

    puntos = np.asarray(
        puntos,
        dtype=np.float64
    )

    arbol = cKDTree(
        puntos
    )

    distancias, _ = (
        arbol.query(
            puntos,
            k=2
        )
    )

    return float(
        np.median(
            distancias[:, 1]
        )
    )


def malla_a_pointcloud_2048(
    ruta_malla,
    semilla
):
    """
    Muestrea 2048 puntos sobre la superficie
    de la predicción Pix2Vox++.
    """

    mesh = trimesh.load(
        ruta_malla,
        force="mesh",
        process=False
    )

    if (
        mesh.is_empty
        or len(mesh.faces) == 0
    ):
        raise RuntimeError(
            "Malla predicha vacía."
        )

    estado = np.random.get_state()

    np.random.seed(
        semilla
    )

    puntos, _ = (
        trimesh.sample.sample_surface(
            mesh,
            2048
        )
    )

    np.random.set_state(
        estado
    )

    return puntos.astype(
        np.float32
    )


def metricas_pointcloud(
    puntos_pred,
    puntos_roto
):
    """
    Compara la nube predicha contra la nube rota real.

    NO utiliza la taza completa.
    """

    puntos_pred = np.asarray(
        puntos_pred,
        dtype=np.float64
    )

    puntos_roto = np.asarray(
        puntos_roto,
        dtype=np.float64
    )


    arbol_roto = cKDTree(
        puntos_roto
    )

    arbol_pred = cKDTree(
        puntos_pred
    )


    # Predicción → roto
    d_pred_roto = (
        arbol_roto.query(
            puntos_pred,
            k=1
        )[0]
    )

    # Roto → predicción
    d_roto_pred = (
        arbol_pred.query(
            puntos_roto,
            k=1
        )[0]
    )


    # ========================================================
    # Chamfer
    # ========================================================

    chamfer = (
        d_pred_roto.mean()
        +
        d_roto_pred.mean()
    )


    # ========================================================
    # Tolerancia geométrica
    # ========================================================

    separacion = (
        separacion_mediana_local(
            puntos_roto
        )
    )

    tolerancia = (
        3.0
        * separacion
    )


    # ========================================================
    # Precision superficial
    #
    # De todo lo que predijo Pix2Vox:
    # ¿cuánto está cerca de la geometría rota?
    # ========================================================

    precision = float(
        np.mean(
            d_pred_roto
            <= tolerancia
        )
    )


    # ========================================================
    # Recall / cobertura
    #
    # De toda la geometría rota:
    # ¿cuánto consigue reconstruir Pix2Vox?
    # ========================================================

    recall = float(
        np.mean(
            d_roto_pred
            <= tolerancia
        )
    )


    # ========================================================
    # F-score
    # ========================================================

    if precision + recall > 0:

        fscore = (
            2
            * precision
            * recall
            / (
                precision
                + recall
            )
        )

    else:
        fscore = 0.0


    # ========================================================
    # Distancia robusta P95
    # ========================================================

    p95_pred_roto = float(
        np.percentile(
            d_pred_roto,
            95
        )
    )

    p95_roto_pred = float(
        np.percentile(
            d_roto_pred,
            95
        )
    )

    hausdorff95 = max(
        p95_pred_roto,
        p95_roto_pred
    )


    return {
        "chamfer_pred_vs_roto":
            float(chamfer),

        "precision_superficie":
            precision,

        "cobertura_rotura":
            recall,

        "fscore_superficie":
            float(fscore),

        "fraccion_pred_fuera_rotura":
            float(
                1.0
                - precision
            ),

        "p95_pred_a_roto":
            p95_pred_roto,

        "p95_roto_a_pred":
            p95_roto_pred,

        "hausdorff95":
            float(hausdorff95),

        "tolerancia":
            float(tolerancia),
    }


# ============================================================
# Ejecutar sobre todas las tazas
# ============================================================

resultados_pc = []


for indice, fila_voxel in tqdm(
    df_validas_voxel.iterrows(),
    total=len(df_validas_voxel),
    desc="Evaluación point cloud"
):

    identificador = (
        fila_voxel[
            "identificador"
        ]
    )

    try:

        fila_datos = (
            df_inferencia[
                df_inferencia[
                    "identificador"
                ] == identificador
            ]
            .iloc[0]
        )


        # ====================================================
        # Predicción Pix2Vox++ → point cloud
        # ====================================================

        puntos_pred = (
            malla_a_pointcloud_2048(
                fila_voxel[
                    "ruta_malla_pred"
                ],
                semilla=(
                    SEMILLA
                    + int(indice)
                )
            )
        )


        ruta_pc = (
            RUTA_PRED_POINTCLOUD
            / f"{identificador}.npy"
        )

        np.save(
            ruta_pc,
            puntos_pred
        )


        # ====================================================
        # Ground truth ROTO
        #
        # Es el *_roto.npy ya llevado al sistema [0,1]^3.
        # ====================================================

        puntos_roto = np.load(
            fila_datos[
                "puntos_render_npy"
            ],
            allow_pickle=False
        ).astype(np.float64)


        # ====================================================
        # Métricas
        # ====================================================

        mets = metricas_pointcloud(
            puntos_pred,
            puntos_roto
        )


        resultados_pc.append({
            "identificador":
                identificador,

            "estado":
                "ok",

            "ruta_pointcloud_pred":
                str(ruta_pc),

            **mets
        })


    except Exception as exc:

        resultados_pc.append({
            "identificador":
                identificador,

            "estado":
                "error",

            "error":
                str(exc)
        })


df_metricas_pointcloud = pd.DataFrame(
    resultados_pc
)


RUTA_METRICAS_PC = (
    RUTA_INFORMES
    / "evaluacion_pix2vox_pointcloud_vs_roto.csv"
)

df_metricas_pointcloud.to_csv(
    RUTA_METRICAS_PC,
    index=False
)


print()
print("=" * 70)
print("MÉTRICAS — POINT CLOUD VS TAZA ROTA")
print("=" * 70)

display(
    df_metricas_pointcloud[
        df_metricas_pointcloud[
            "estado"
        ] == "ok"
    ][
        [
            "chamfer_pred_vs_roto",
            "precision_superficie",
            "cobertura_rotura",
            "fscore_superficie",
            "fraccion_pred_fuera_rotura",
            "hausdorff95",
        ]
    ].describe()
)

print()
print(
    "CSV:",
    RUTA_METRICAS_PC
)

In [ ]:
# ============================================================
# 20C. COMPROBAR QUÉ FONDO TIENEN REALMENTE LOS PNG
#     Y QUÉ RGB ESTAMOS PASANDO A PIX2VOX++
# ============================================================

resultados_alpha = []


for _, resultado in df_20_visual.iterrows():

    identificador = (
        resultado[
            "identificador"
        ]
    )

    fila = (
        df_inferencia.loc[
            df_inferencia[
                "identificador"
            ] == identificador
        ]
        .iloc[0]
    )


    for columna in COLUMNAS_VISTAS:

        ruta = Path(
            fila[columna]
        )

        imagen_original = (
            Image.open(
                ruta
            )
        )

        rgba = np.asarray(
            imagen_original.convert(
                "RGBA"
            )
        )

        rgb_modelo = np.asarray(
            imagen_original.convert(
                "RGB"
            )
        )


        alpha = (
            rgba[:, :, 3]
        )

        fondo = (
            alpha == 0
        )

        objeto = (
            alpha > 0
        )


        if fondo.any():

            rgb_fondo_medio = (
                rgb_modelo[
                    fondo
                ]
                .mean(
                    axis=0
                )
            )

        else:

            rgb_fondo_medio = (
                np.array(
                    [
                        np.nan,
                        np.nan,
                        np.nan
                    ]
                )
            )


        if objeto.any():

            rgb_objeto_medio = (
                rgb_modelo[
                    objeto
                ]
                .mean(
                    axis=0
                )
            )

        else:

            rgb_objeto_medio = (
                np.array(
                    [
                        np.nan,
                        np.nan,
                        np.nan
                    ]
                )
            )


        resultados_alpha.append({

            "identificador":
                identificador,

            "vista":
                columna,

            "modo_original":
                imagen_original.mode,

            "fraccion_transparente":
                float(
                    fondo.mean()
                ),

            "fondo_R":
                float(
                    rgb_fondo_medio[0]
                ),

            "fondo_G":
                float(
                    rgb_fondo_medio[1]
                ),

            "fondo_B":
                float(
                    rgb_fondo_medio[2]
                ),

            "objeto_R":
                float(
                    rgb_objeto_medio[0]
                ),

            "objeto_G":
                float(
                    rgb_objeto_medio[1]
                ),

            "objeto_B":
                float(
                    rgb_objeto_medio[2]
                ),
        })


df_alpha = pd.DataFrame(
    resultados_alpha
)


print("=" * 80)
print("DIAGNÓSTICO DEL FONDO DE LAS IMÁGENES")
print("=" * 80)

display(
    df_alpha.head(
        20
    ).round(
        2
    )
)


print()
print("PROMEDIOS:")

display(
    df_alpha[
        [
            "fraccion_transparente",
            "fondo_R",
            "fondo_G",
            "fondo_B",
            "objeto_R",
            "objeto_G",
            "objeto_B",
        ]
    ]
    .mean()
    .to_frame(
        "media"
    )
    .round(
        2
    )
)

# Pipeline

Cuando el pipeline esté validado, cada taza rota tendrá:

```text
Fantastik_Break_Pix2Vox/z
├── mallas_bpa/<id>.ply
├── puntos_render/<id>.npy
├── renders_5v/<id>/
│   ├── camera_poses.json
│   └── rendering/
│       ├── 00.png
│       ├── 05.png
│       ├── 10.png
│       ├── 14.png
│       └── 19.png
└── informes/catalogo_fantastik_roto_5v.csv
```

El siguiente notebook podrá cargar los checkpoints de los experimentos seleccionados y evaluar las predicciones frente al `*_roto.npy` original mediante métricas de superficie, especialmente Chamfer Distance y F-score.

Para E3 debe confirmarse qué representación NPY consume el modelo:

- `(32,32,32)` si trabaja con voxels;
- `(2048,3)` si trabaja con point clouds como Fantastik.